
![DBAcademy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/databricks_academy.png)

# Lecture - Streaming Joins and Deploying Pipelines to Production

## Overview

In this lecture, you will learn the fundamentals of streaming joins in Apache Spark™ Declarative Pipelines, including stream-snapshot joins, joins through materialized views, and stream-stream joins. You will also learn how to operationalize a pipeline for production using scheduling, notifications, monitoring, and event logs.

## Learning Objectives

By the end of this lecture, you will be able to:

- **Deploy** a Apache Spark™ Declarative Pipeline in production by modifying configuration options like mode, schedule, email notifications and more
- **Analyze** event logs and pipeline metrics to examine the entirety of a pipeline


## A. What Are Streaming Joins?

When performing joins with streaming tables in Declarative Pipelines, it is important to understand the different join types and how they behave.

<div style="max-width:1200px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
  <div style="margin:20px 0;">
    <div style="font-size:14pt; font-weight:800; color:#618794; text-transform:uppercase; letter-spacing:0.06em; margin-bottom:16px; text-align:center;">
      Join Types Covered
    </div>
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(260px, 1fr)); gap:16px; margin:20px 0;">
    <!-- Block 1 -->
    <div style="background:#F9F7F4; border-radius:12px; padding:20px 22px; border:2px solid #2272B4; box-shadow:0 3px 12px rgba(34,114,180,0.10);">
      <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:14px; border-bottom:2px solid #EEEDE9; padding-bottom:10px;">
        <div style="font-weight:800; font-size:12pt; color:#0B2026;">Stream-Snapshot Join</div>
        <div style="font-size:10pt; color:#618794; margin-top:3px;">Streaming table joined with a static table
</div>
      </div>
      <div style="display:flex; align-items:flex-start; gap:14px; margin-bottom:14px;">
        <div style="font-size:10.5pt; color:#1B3139; line-height:1.8; flex:1;">
          The goal is to <strong>incrementally join new data</strong> from a streaming table with a
          <strong>static lookup table</strong> to create another streaming table.
        </div>
        <img
          src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_streaming_joins/stream_snapshot_goal.png"
          alt="Stream-Snapshot Join Diagram"
          style="width:120px; height:auto; border-radius:8px; border:1.5px solid #DCE0E2; flex-shrink:0; object-fit:contain; background:white;">
      </div>
    </div>
    <!-- Block 2 -->
    <div style="background:#F9F7F4; border-radius:12px; padding:20px 22px; border:2px solid #2272B4; box-shadow:0 3px 12px rgba(34,114,180,0.10);">
      <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:14px; border-bottom:2px solid #EEEDE9; padding-bottom:10px;">
        <div style="font-weight:800; font-size:12pt; color:#0B2026;">Streaming via Materialized View</div>
          <div style="font-size:10pt; color:#618794; margin-top:3px;">Two streaming tables joined in a materialized view</div>
      </div>
      <div style="display:flex; align-items:flex-start; gap:14px; margin-bottom:14px;">
        <div style="font-size:10.5pt; color:#1B3139; line-height:1.8; flex:1;">
        The goal is to take <strong>all rows from two streaming tables</strong> and join them together each time the pipeline is run.
      </div>
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_streaming_joins/streaming_table_goal.png" alt="Streaming Table Join Diagram"
        style="width:120px; height:auto; border-radius:8px; border:1.5px solid #DCE0E2; flex-shrink:0; object-fit:contain;">
    </div>
    </div>
    <!-- Block 3 -->
    <div style="background:#F9F7F4; border-radius:12px; padding:20px 22px; border:2px solid #2272B4; box-shadow:0 3px 12px rgba(34,114,180,0.10);">
      <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:14px; border-bottom:2px solid #EEEDE9; padding-bottom:10px;">
        <div style="font-weight:800; font-size:12pt; color:#0B2026;">Stream-Stream Join</div>
          <div style="font-size:10pt; color:#618794; margin-top:3px;">Incremental join of two live streams (advanced)</div>
      </div>
      <div style="display:flex; align-items:flex-start; gap:14px; margin-bottom:14px;">
        <div style="font-size:10.5pt; color:#1B3139; line-height:1.8; flex:1;">
        The goal is to <b>incrementally</b> join new data from two tables, <b>past data is not used</b>
      </div>
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_streaming_joins/stream_stream_joins.png" alt="Stream-Stream Join Diagram"
        style="width:120px; height:auto; border-radius:8px; border:1.5px solid #DCE0E2; flex-shrink:0; object-fit:contain;">
    </div>
    </div>
  </div>
</div>

#####EXPAND FOR ADDITIONAL NOTES
<details>

<b>Stream-Snapshot Join Overview</b>

Let’s start with the most straightforward: Joining a Streaming Table to a Static Table (sometimes called a Stream-Snapshot Join or Stream-static join depending on the scenario)

In this pattern, we incrementally join new data from a streaming table to a static lookup table to create another streaming table.

For example, imagine you’re ingesting a stream of transactions that includes a country_code column. You can join that with a static reference table that maps each code to a full country name.

This is useful when you want to enrich your streaming data with reference information that doesn’t change often.

<b>Joining Streaming Tables with a Materialized View</b>

Another supported pattern is joining two streaming tables using a materialized view.

The goal here is to join all rows from both streaming tables each time the pipeline is run.

This type of join is typically used when both input datasets are changing continuously, and you need to combine them regularly to produce a unified, up-to-date result.

For example, imagine one stream contains customer activity and another contains product catalog updates. You could join them using a materialized view to enrich activity data with the most recent product information.

Because both sides are streaming, a materialized view is required to handle this join efficiently and keep the results current.

The materialized view will process all new rows from both tables and incrementally refresh, depending on pipeline configuration and compute mode.

<b>Stream-Stream Joins</b>

The final type of streaming join is a stream-stream join, which is designed to incrementally join new data from two streaming tables as it arrives.

In this pattern, only the new incoming data from each stream is joined, past data is not considered during each run.

These joins are useful for detecting relationships between events that occur close together in time, such as joining clickstream data with real-time ad impressions.

However, because stream-stream joins often involve windowing logic, watermarking, and other advanced streaming concepts, they are outside the scope of this course.

We recommend reviewing additional Databricks documentation or advanced streaming resources if your use case requires this type of real-time event correlation.

</details>


## B. Stream-Snapshot Join Overview

Let’s walk through how this type of join actually works in practice.

As new data arrives in your streaming table, it is joined in real time with the static reference table.


<div style="max-width: 1000px; margin: 0 auto; font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

  <style>
    .ssj-wrap {
      display: flex;
      align-items: flex-start;
      justify-content: center;
      gap: 32px;
      margin: 20px 0;
    }

    .ssj-col {
      display: flex;
      flex-direction: column;
      align-items: center;
      gap: 6px;
    }

    .ssj-col img {
      width: 56px;
      height: 40px;
      object-fit: contain;
      background: transparent;
      mix-blend-mode: multiply;
      filter: contrast(1.05) brightness(1.02);
    }

    .ssj-title-out {
      font-size: 12pt;
      font-weight: 700;
      color: #0B2026;
      text-align: center;
    }

    .ssj-card {
      width: 260px;
      height: 210px;
      border-radius: 6px;
      border: 2px solid #1B3139;  /* black border box */
      background: #FFFFFF;
      box-sizing: border-box;
      padding: 12px 14px;
    }

    .ssj-card.small {
      width: 230px;
    }

    /* Streaming table inner dashed box + Append (top aligned) */
    .ssj-append-label {
      font-size: 11pt;
      font-weight: 700;
      color:#0B2026;
      margin-bottom:6px;
      text-align:center;
    }

    .ssj-append-area {
      border-radius: 4px;
      border: 1.5px dashed #C4CCD6;
      padding: 8px 8px 10px 8px;
      box-sizing: border-box;
    }

    .ssj-bar-blue {
      height: 10px;
      background:#4299E0;
      border-radius:3px;
      margin-bottom:5px;
    }
    .ssj-bar-blue:last-child { margin-bottom:0; }

    /* Joined streaming table rows */
    .ssj-row-joined {
      display:flex;
      margin-bottom:5px;
      border-radius:3px;
      overflow:hidden;
      height:10px;
    }
    .ssj-row-joined .left {
      flex:1;
      background:#4299E0;
    }
    .ssj-row-joined .right {
      flex:1;
      background:#00A972;
    }
    .ssj-row-joined:last-child { margin-bottom:0; }

    /* Static table bars */
    .ssj-static-bar {
      height:10px;
      background:#00A972;
      border-radius:3px;
      margin-bottom:4px;
    }
    .ssj-static-bar:last-child { margin-bottom:0; }

    /* Connectors */
    .ssj-arrow-horizontal {
      width: 80px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
    }
    .ssj-arrow-horizontal::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    .ssj-arrow-to-join {
      width: 80px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
      margin-left: -8px;
    }
    .ssj-arrow-to-join::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    /* Ingest arrow */
    .sj-ingest-arrow {
      width: 100px;
      height: 40px;
      border-radius: 6px;
      background: #4299E0;
      clip-path: polygon(0 20%, 60% 20%, 60% 0, 100% 50%, 60% 100%, 60% 80%, 0 80%);
      margin-top: 100px;
    }
  </style>

  <div class="ssj-wrap">
    <!-- Left: ingest arrow using CSS shape -->
    <div class="sj-ingest-arrow"></div>
    <!-- Streaming Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
      <div class="ssj-title-out">Streaming Table</div>
      <!-- Black border box with dashed inner box + Append at top -->
      <div class="ssj-card">
        <div class="ssj-append-area">
          <div class="ssj-bar-blue"></div>
          <div class="ssj-bar-blue"></div>
          <div class="ssj-bar-blue"></div>
        </div>
        <div class="ssj-append-label">Append</div>
      </div>
    </div>
    <!-- Arrow to Joined Streaming Table -->
    <div class="ssj-arrow-horizontal"></div>
    <!-- Joined Streaming Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
      <div class="ssj-title-out">Joined Streaming Table</div>
      <!-- Black border box -->
      <div class="ssj-card">
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
      </div>
    </div>
    <!-- Arrow from Static Table into Joined Streaming Table -->
    <div class="ssj-arrow-to-join"></div>
    <!-- Static Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/static_table_icon.png" alt="Static table icon">
      <div class="ssj-title-out">Static Table</div>
      <!-- Black border box -->
      <div class="ssj-card small">
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
      </div>
    </div>
  </div>
</div>

#####EXPAND FOR ADDITIONAL NOTES
<details>

That means every new record in the streaming table is matched against all available data in the static table, ensuring a complete and accurate join each time.

The result of that join is then appended to your final streaming output table, continuing the pipeline flow.

This is a great pattern for enriching streaming data with context, like converting country codes, product IDs, or user roles into readable formats using static reference data.

It’s efficient, reliable, and works well since only one side of the join, the streaming table, is changing over time.

</details>

A key detail to understand here is that as new data is appended to your source streaming table, only the new rows are joined with the entire static lookup table.

<div style="max-width: 1100px; margin: 0 auto; font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

  <style>
    .ssj-wrap {
      display: flex;
      align-items: flex-start;
      justify-content: center;
      gap: 32px;
      margin: 20px 0;
    }

    .ssj-col {
      display: flex;
      flex-direction: column;
      align-items: center;
      gap: 6px;
    }

    .ssj-col img {
      width: 56px;
      height: 40px;
      object-fit: contain;
      background: transparent;
      mix-blend-mode: multiply;
      filter: contrast(1.05) brightness(1.02);
    }

    .ssj-title-out {
      font-size: 12pt;
      font-weight: 700;
      color: #0B2026;
      text-align: center;
    }

    .ssj-card {
      width: 260px;
      height: 210px;
      border-radius: 6px;
      border: 2px solid #1B3139;
      background: #FFFFFF;
      box-sizing: border-box;
      padding: 12px 14px;
    }

    .ssj-card.small {
      width: 230px;
    }

    /* Streaming table inner dashed box + Append */
    .ssj-append-area {
      border-radius: 4px;
      border: 1.5px dashed #C4CCD6;
      padding: 8px 8px 10px 8px;
      box-sizing: border-box;
      margin-top: 46px;
    }

    .ssj-bar-blue {
      height: 10px;
      background:#4299E0;
      border-radius:3px;
      margin-bottom:5px;
    }
    .ssj-bar-blue:last-child { margin-bottom:0; }

    .ssj-append-label {
      font-size: 11pt;
      font-weight: 700;
      color:#0B2026;
      margin-top:10px;
      text-align:center;
    }

    /* Grey header rows in tables */
    .ssj-grey-row {
      height:10px;
      background:#D3D6DA;
      border-radius:3px;
      margin-bottom:4px;
    }

    /* Joined streaming table rows */
    .ssj-row-joined {
      display:flex;
      margin-bottom:5px;
      border-radius:3px;
      overflow:hidden;
      height:10px;
    }
    .ssj-row-joined .left {
      flex:1;
      background:#4299E0;
    }
    .ssj-row-joined .right {
      flex:1;
      background:#00A972;
    }
    .ssj-row-joined:last-child { margin-bottom:0; }

    /* Static table bars */
    .ssj-static-bar {
      height:10px;
      background:#00A972;
      border-radius:3px;
      margin-bottom:4px;
    }
    .ssj-static-bar:last-child { margin-bottom:0; }

    /* Connectors */
    .ssj-arrow-horizontal {
      width: 80px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
    }
    .ssj-arrow-horizontal::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    .ssj-arrow-to-join {
      width: 80px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
      margin-left: -8px;
    }
    .ssj-arrow-to-join::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    /* Ingest arrow */
    .sj-ingest-arrow {
      width: 100px;
      height: 40px;
      border-radius: 6px;
      background: #4299E0;
      clip-path: polygon(0 20%, 60% 20%, 60% 0, 100% 50%, 60% 100%, 60% 80%, 0 80%);
      margin-top: 100px;
    }
  </style>

  <div class="ssj-wrap">
    <!-- Left: ingest arrow using CSS shape -->
    <div class="sj-ingest-arrow"></div>
    <!-- Streaming Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
      <div class="ssj-title-out">Streaming Table</div>
      <!-- Card -->
      <div class="ssj-card">
        <div class="ssj-grey-row"></div>
        <div class="ssj-grey-row"></div>
        <div class="ssj-grey-row" style="margin-bottom:10px;"></div>
        <div class="ssj-append-area">
          <div class="ssj-bar-blue"></div>
          <div class="ssj-bar-blue"></div>
          <div class="ssj-bar-blue"></div>
        </div>
        <div class="ssj-append-label">Append</div>
      </div>
    </div>
    <!-- Arrow to Joined Streaming Table -->
    <div class="ssj-arrow-horizontal"></div>
    <!-- Joined Streaming Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Joined streaming table icon">
      <div class="ssj-title-out">Joined Streaming Table</div>
      <!-- Card -->
      <div class="ssj-card">
        <div class="ssj-grey-row"></div>
        <div class="ssj-grey-row"></div>
        <div class="ssj-grey-row" style="margin-bottom:10px;"></div>
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
        <div class="ssj-row-joined">
          <div class="left"></div>
          <div class="right"></div>
        </div>
      </div>
    </div>
    <!-- Arrow from Static Table into Joined Streaming Table -->
    <div class="ssj-arrow-to-join"></div>
    <!-- Static Table -->
    <div class="ssj-col">
      <!-- Icon and name OUTSIDE the box -->
      <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/static_table_icon.png" alt="Static table icon">
      <div class="ssj-title-out">Static Table</div>
      <!-- Card -->
      <div class="ssj-card small">
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
        <div class="ssj-static-bar"></div>
      </div>
    </div>
  </div>
</div>

#####EXPAND FOR ADDITIONAL NOTES

<details>

This is an incremental join, meaning the static table doesn’t need to be reprocessed, only the incoming streaming data is evaluated.

This keeps the process highly efficient, and ensures that each new record is immediately enriched with the latest reference data from the static table.

</details>

#####Documentation

- <a href="https://docs.databricks.com/gcp/en/transform/join#stream-stream-joins" style="color: #1976d2; text-decoration: underline;">Stream-snapshot joins</a>

## C. Joining Streaming Tables with a Materialized View

Let’s walk through how this join actually works in practice.

As new data is appended to both streaming tables, the materialized view detects the changes and responds accordingly.

<div style="max-width: 1100px; margin: 0 auto; font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

  <style>
    .mv-wrap {
      display: flex;
      align-items: flex-start;
      justify-content: center;
      gap: 32px;
      margin: 24px 0;
    }

    .mv-col {
      display: flex;
      flex-direction: column;
      align-items: center;
      gap: 6px;
    }

    .mv-col img {
      width: 56px;
      height: 40px;
      object-fit: contain;
      background: transparent;
      mix-blend-mode: multiply;
      filter: contrast(1.05) brightness(1.02);
    }

    .mv-title {
      font-size: 13pt;
      font-weight: 700;
      color: #0B2026;
      text-align: center;
    }

    .mv-card {
      width: 280px;
      height: 220px;
      border-radius: 6px;
      border: 2px solid #1B3139;
      background: #FFFFFF;
      box-sizing: border-box;
      padding: 12px 16px;
    }

    /* Dashed “streaming” area */
    .mv-dashed {
      border-radius: 4px;
      border: 1.5px dashed #C4CCD6;
      padding: 8px 8px 10px 8px;
      box-sizing: border-box;
      margin-bottom: 10px;
      margin-top: 4px;
    }

    .mv-bar-blue {
      height: 10px;
      background:#4299E0;
      border-radius:3px;
      margin-bottom:5px;
    }
    .mv-bar-blue:last-child { margin-bottom:0; }

    .mv-bar-yellow {
      height: 10px;
      background:#FFB020;
      border-radius:3px;
      margin-bottom:5px;
    }
    .mv-bar-yellow:last-child { margin-bottom:0; }

    .mv-append {
      font-size: 12pt;
      font-weight: 700;
      color:#0B2026;
      text-align:center;
      margin-top: 4px;
    }

    /* Joined MV rows: half blue, half yellow */
    .mv-row-joined {
      display:flex;
      margin-bottom:6px;
      border-radius:3px;
      overflow:hidden;
      height:10px;
      margin-top: 8px;
    }
    .mv-row-joined .left {
      flex:1;
      background:#4299E0;
    }
    .mv-row-joined .right {
      flex:1;
      background:#FFB020;
    }
    .mv-row-joined:last-child { margin-bottom:0; }

    /* Connector arrows */
    .mv-arrow-horizontal {
      width: 90px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
    }
    .mv-arrow-horizontal::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    /* Ingest arrow */
    .sj-ingest-arrow {
      width: 100px;
      height: 40px;
      border-radius: 6px;
      background: #4299E0;
      clip-path: polygon(0 20%, 60% 20%, 60% 0, 100% 50%, 60% 100%, 60% 80%, 0 80%);
      margin-top: 100px;
    }

    /* Side ingest arrows */
    .mv-arrow-left {
      width: 80px;
      height: 40px;
      border-radius: 6px;
      background: #4299E0;
      clip-path: polygon(0 50%, 40% 0, 40% 25%, 100% 25%, 100% 75%, 40% 75%, 40% 100%);
      margin-top: 110px;
    }

    .mv-arrow-right {
      width: 80px;
      height: 40px;
      border-radius: 6px;
      background: #FFB020;
      clip-path: polygon(0 25%, 60% 25%, 60% 0, 100% 50%, 60% 100%, 60% 75%, 0 75%);
      margin-top: 110px;
    }
  </style>

  <div style="display:flex; align-items:flex-start; justify-content:center; gap:20px;">
    <!-- Left outer arrow -->
    <div class="mv-arrow-left"></div>
    <div class="mv-wrap">
      <!-- Left Streaming Table -->
      <div class="mv-col">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
        <div class="mv-title">Streaming Table</div>
        <div class="mv-card">
          <div class="mv-dashed">
            <div class="mv-bar-blue"></div>
            <div class="mv-bar-blue"></div>
            <div class="mv-bar-blue"></div>
          </div>
          <div class="mv-append">Append</div>
        </div>
      </div>
      <!-- Arrow to Joined MV -->
      <div class="mv-arrow-horizontal"></div>
      <!-- Joined Materialized View -->
      <div class="mv-col">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/materialized.png" alt="Joined materialized view icon">
        <div class="mv-title">Joined Materialized View</div>
        <div class="mv-card">
          <div class="mv-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
        </div>
      </div>
      <!-- Arrow back to right Streaming Table -->
      <div class="mv-arrow-horizontal" style="transform:scaleX(-1);"></div>
      <!-- Right Streaming Table -->
      <div class="mv-col">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
        <div class="mv-title">Streaming Table</div>
        <div class="mv-card">
          <div class="mv-dashed">
            <div class="mv-bar-yellow"></div>
            <div class="mv-bar-yellow"></div>
            <div class="mv-bar-yellow"></div>
          </div>
          <div class="mv-append">Append</div>
        </div>
      </div>
    </div>
    <!-- Right outer arrow -->
    <div class="mv-arrow-right"></div>
  </div>
</div>

#####EXPAND FOR ADDITIONAL NOTES
<details>

Each time the pipeline runs, the materialized view will efficiently compute the join by processing all the current data from both streaming tables.

This ensures that every new record from either side is matched correctly, producing an up-to-date and complete output.

The use of a materialized view allows the join to scale efficiently, leveraging incremental refresh where possible to avoid unnecessary recomputation.

This join pattern is ideal when both data sources are live and frequently updated, and you want to keep your results synchronized across both.

</details>

As new data is added to the streaming tables, the materialized view will again efficiently compute all the data in the streaming tables and joins the rows.


<div style="max-width: 1100px; margin: 0 auto; font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

  <style>
    .mv2-wrap {
      display: flex;
      align-items: flex-start;
      justify-content: center;
      gap: 32px;
      margin: 24px 0;
    }

    .mv2-col {
      display: flex;
      flex-direction: column;
      align-items: center;
      gap: 6px;
    }

    .mv2-col img {
      width: 56px;
      height: 40px;
      object-fit: contain;
      background: transparent;
      mix-blend-mode: multiply;
      filter: contrast(1.05) brightness(1.02);
    }

    .mv2-title {
      font-size: 13pt;
      font-weight: 700;
      color: #0B2026;
      text-align: center;
    }

    .mv2-card {
      width: 260px;
      height: 210px;
      border-radius: 6px;
      border: 2px solid #1B3139;
      background: #FFFFFF;
      box-sizing: border-box;
      padding: 10px 14px 14px 14px;
    }

    .mv2-card.small {
      width: 260px;
    }

    /* Dashed areas for streaming tables */
    .mv2-dashed {
      border-radius: 4px;
      border: 1.5px dashed #C4CCD6;
      padding: 8px 8px 10px 8px;
      box-sizing: border-box;
      margin-bottom: 8px;
    }

    .mv2-bar-blue {
      height: 10px;
      background:#4299E0;
      border-radius:3px;
      margin-bottom:5px;
    }
    .mv2-bar-blue:last-child { margin-bottom:0; }

    .mv2-bar-yellow {
      height: 10px;
      background:#FFB020;
      border-radius:3px;
      margin-bottom:5px;
    }
    .mv2-bar-yellow:last-child { margin-bottom:0; }

    .mv2-append {
      font-size: 12pt;
      font-weight: 700;
      color:#0B2026;
      text-align:center;
      margin-top: 4px;
    }

    /* Joined MV rows: half blue, half yellow */
    .mv2-row-joined {
      display:flex;
      margin-bottom:6px;
      border-radius:3px;
      overflow:hidden;
      height:10px;
      margin-top: 4px;
    }
    .mv2-row-joined .left {
      flex:1;
      background:#4299E0;
    }
    .mv2-row-joined .right {
      flex:1;
      background:#FFB020;
    }
    .mv2-row-joined:last-child { margin-bottom:0; }

    /* Connectors between boxes */
    .mv2-arrow-horizontal {
      width: 90px;
      height: 0;
      border-top: 2px solid #1B3139;
      position: relative;
      margin-top: 120px;
    }
    .mv2-arrow-horizontal::after {
      content:"";
      position:absolute;
      right:-1px;
      top:-6px;
      border-top:5px solid transparent;
      border-bottom:5px solid transparent;
      border-left:8px solid #1B3139;
    }

    /* Outer ingest arrows */
    .mv2-arrow-left {
      width: 100px;
      height: 40px;
      border-radius: 6px;
      background: #4299E0;
      clip-path: polygon(0 20%, 60% 20%, 60% 0, 100% 50%, 60% 100%, 60% 80%, 0 80%);
      margin-top: 110px;
    }

    .mv2-arrow-right {
      width: 100px;
      height: 40px;
      border-radius: 6px;
      background: #FFB020;
      clip-path: polygon(0 20%, 60% 20%, 60% 0, 100% 50%, 60% 100%, 60% 80%, 0 80%);
      margin-top: 110px;
      transform: scaleX(-1);
    }
  </style>

  <div style="display:flex; align-items:flex-start; justify-content:center; gap:20px;">
    <!-- Left outer arrow -->
    <div class="mv2-arrow-left"></div>
    <div class="mv2-wrap">
      <!-- Left Streaming Table (blue) -->
      <div class="mv2-col">
        <!-- Replace src with streaming-table icon -->
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
        <div class="mv2-title">Streaming Table</div>
        <div class="mv2-card">
          <div class="mv2-dashed">
            <div class="mv2-bar-blue"></div>
            <div class="mv2-bar-blue"></div>
          </div>
          <div class="mv2-dashed">
            <div class="mv2-bar-blue"></div>
            <div class="mv2-bar-blue"></div>
          </div>
          <div class="mv2-append">Append</div>
        </div>
      </div>
      <!-- Arrow to Joined Materialized View -->
      <div class="mv2-arrow-horizontal"></div>
      <!-- Joined Materialized View -->
      <div class="mv2-col">
        <!-- Replace src with materialized-view icon -->
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/materialized.png" alt="Joined materialized view icon">
        <div class="mv2-title">Joined Materialized View</div>
        <div class="mv2-card small">
          <div class="mv2-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv2-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv2-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv2-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
          <div class="mv2-row-joined">
            <div class="left"></div>
            <div class="right"></div>
          </div>
        </div>
      </div>
      <!-- Arrow from MV back to right Streaming Table -->
      <div class="mv2-arrow-horizontal" style="transform:scaleX(-1);"></div>
      <!-- Right Streaming Table (yellow) -->
      <div class="mv2-col">
        <!-- Replace src with streaming-table icon -->
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/streaming.png" alt="Streaming table icon">
        <div class="mv2-title">Streaming Table</div>
        <div class="mv2-card">
          <div class="mv2-dashed">
            <div class="mv2-bar-yellow"></div>
            <div class="mv2-bar-yellow"></div>
          </div>
          <div class="mv2-dashed">
            <div class="mv2-bar-yellow"></div>
            <div class="mv2-bar-yellow"></div>
          </div>
          <div class="mv2-append">Append</div>
        </div>
      </div>
    </div>
    <!-- Right outer arrow -->
    <div class="mv2-arrow-right"></div>
  </div>
</div>

## D. Stream-Stream Joins

The final type of streaming join is a stream-stream join, which is designed to incrementally join new data from two streaming tables as it arrives.

- In this pattern, only the new incoming data from each stream is joined, past data is not considered during each run.

- These joins are useful for detecting relationships between events that occur close together in time, such as joining clickstream data with real-time ad impressions.

However, because stream-stream joins often involve windowing logic, watermarking, and other advanced streaming concepts, they are outside the scope of this course.

We recommend reviewing additional Databricks documentation or advanced streaming resources if your use case requires this type of real-time event correlation.


#####Documentation

- <a href="https://docs.databricks.com/gcp/en/transform/join#stream-stream-joins" style="color: #1976d2; text-decoration: underline;">Stream-stream joins</a>
- <a href="https://docs.databricks.com/aws/en/dlt/stateful-processing"  style="color: #1976d2; text-decoration: underline;">Optimize stateful processing in Apache Spark™ Declarative Pipelines with watermarks</a>

## E. Comparing the Join Types

The table below summarizes all three join patterns at a glance. Use this as a quick reference when deciding which join type fits your pipeline's requirements.

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">

<!-- Comparison Table -->
<div style="border-radius:12px; overflow:hidden; border:2px solid #EEEDE9; box-shadow:0 3px 12px rgba(0,0,0,0.06); margin:20px 0;">
  <div style="background:#0B2026; padding:12px 20px; color:white; font-weight:800; font-size:11pt;">All Three Join Types — At a Glance</div>
  <table style="width:100%; border-collapse:collapse; font-family:'Segoe UI',sans-serif;">
    <thead>
      <tr style="background:#F9F7F4;">
        <th style="padding:12px 16px; text-align:left; font-size:9.5pt; color:#618794; text-transform:uppercase; letter-spacing:0.06em; border-bottom:2px solid #EEEDE9;">Join Type</th>
        <th style="padding:12px 16px; text-align:left; font-size:9.5pt; color:#618794; text-transform:uppercase; letter-spacing:0.06em; border-bottom:2px solid #EEEDE9;">Sources</th>
        <th style="padding:12px 16px; text-align:left; font-size:9.5pt; color:#618794; text-transform:uppercase; letter-spacing:0.06em; border-bottom:2px solid #EEEDE9;">Output Type</th>
        <th style="padding:12px 16px; text-align:left; font-size:9.5pt; color:#618794; text-transform:uppercase; letter-spacing:0.06em; border-bottom:2px solid #EEEDE9;">Data Processed</th>
        <th style="padding:12px 16px; text-align:left; font-size:9.5pt; color:#618794; text-transform:uppercase; letter-spacing:0.06em; border-bottom:2px solid #EEEDE9;">In Scope?</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom:1px solid #EEEDE9;">
        <td style="padding:12px 16px;"><span style="background:#2272B4; color:white; font-weight:800; padding:3px 12px; border-radius:999px; font-size:9.5pt;">Stream-Snapshot</span></td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Streaming + Static</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Streaming Table</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">New rows only</td>
        <td style="padding:12px 16px; color:#00A972; font-weight:700; font-size:10pt;">Yes</td>
      </tr>
      <tr style="border-bottom:1px solid #EEEDE9; background:#F9F7F4;">
        <td style="padding:12px 16px;"><span style="background:#00A972; color:white; font-weight:800; padding:3px 12px; border-radius:999px; font-size:9.5pt;">MV Join</span></td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Streaming + Streaming</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Materialized View</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">All rows each run</td>
        <td style="padding:12px 16px; color:#00A972; font-weight:700; font-size:10pt;">Yes</td>
      </tr>
      <tr>
        <td style="padding:12px 16px;"><span style="background:#618794; color:white; font-weight:800; padding:3px 12px; border-radius:999px; font-size:9.5pt;">Stream-Stream</span></td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Streaming + Streaming</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">Streaming Table</td>
        <td style="padding:12px 16px; color:#1B3139; font-size:10pt;">New rows only (windowed)</td>
        <td style="padding:12px 16px; color:#98102A; font-weight:700; font-size:10pt;">Advanced only</td>
      </tr>
    </tbody>
  </table>
</div>
</div>

## F. The Complete Pipeline — What We Have Built

<div style="text-align: center; margin-top: 20px;">
  <img
    src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_deploying_pipeline_production/order_status_flow_pipeline.png"
    alt="LakeFlow Connect Unified Ingestion"
    style="width: 1200px; max-width: 100%; height: auto;">
</div>

#####EXPAND FOR ADDITIONAL NOTES
<details>

So far, we’ve completed the first flow of our pipeline:

Orders Flow: <b>orders JSON files → orders_bronze → orders_silver → gold_orders_by_date</b>

Now we’re ready to add the second flow to the pipeline, the Status flow.

Here’s what this new flow will do:

- Ingest status JSON files to the status_bronze streaming table
- Transform the bronze streaming table into the status_silver streaming table
- Join status_silver with orders_silver to create a new materialized view: full_order_info_gold
- From there, we’ll build two additional materialized views:
<ul>
<li>cancelled_orders
<li>Delivered_orders
</ul>

This flow gives us rich insight into order lifecycle states, and demonstrates how we can join flows together and branch logic for more advanced analytics.

After we complete the Status Flow, our next step is to move the pipeline from development to production.

</details>

## G. Schedule, Notifications, and Monitoring

Use these key steps to operationalize the pipeline. When you're ready to move your Apache Spark™ Declarative Pipeline into production, there are four key operational tasks to ensure it's reliable, automated, and monitored:

<div style="max-width: 1100px; margin: 20px auto 0 auto; font-family: sans-serif; color: #0b2026;">
  <div style="display: flex; gap: 16px; align-items: stretch;">
    <!-- Schedule the Pipeline -->
    <div style="flex: 1; border: 2px solid #FF5F46; border-radius: 10px; padding: 20px;">
      <div style="text-align: center; margin-bottom: 12px;">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/scheduling_icon.png"
             alt="Schedule Pipeline icon"
             style="max-height: 80px; max-width: 100px; width: auto; height: auto;">
      </div>
      <div style="font-size: 16pt; font-weight: 700; margin-bottom: 12px;">Schedule the Pipeline</div>
      <ul style="font-size: 14pt; line-height: 1.7; padding-left: 18px; margin: 0;">
        <li>Define a schedule for automatic execution (e.g., every 15 minutes, hourly, daily, continuous).</li>
        <li>This ensures your data is refreshed regularly without manual triggers.</li>
      </ul>
    </div>
    <!-- Set Up Email Notifications -->
    <div style="flex: 1; border: 2px solid #FF5F46; border-radius: 10px; padding: 20px;">
      <div style="text-align: center; margin-bottom: 12px;">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/email_notifications_icon.png"
             alt="Email Notifications icon"
             style="max-height: 80px; max-width: 100px; width: auto; height: auto;">
      </div>
      <div style="font-size: 16pt; font-weight: 700; margin-bottom: 12px;">Set Up Email Notifications</div>
      <div style="font-size: 14pt; line-height: 1.6;">
        Configure alerts to notify you or your team of:
        <ul style="padding-left: 18px; margin: 4px 0 10px;">
          <li>Successful runs</li>
          <li>Failures</li>
          <li>Starts</li>
        </ul>
        <div>Helps with proactive monitoring and response.</div>
      </div>
    </div>
    <!-- Monitor with Event Log -->
    <div style="flex: 1; border: 2px solid #FF5F46; border-radius: 10px; padding: 20px;">
      <div style="text-align: center; margin-bottom: 12px;">
        <img src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/icons/monitor_pipelines_icon.png"
             alt="Event Log icon"
             style="max-height: 80px; max-width: 100px; width: auto; height: auto;">
      </div>
      <div style="font-size: 16pt; font-weight: 700; margin-bottom: 12px;">Monitor with the Event Log</div>
      <ul style="font-size: 14pt; line-height: 1.7; padding-left: 18px; margin: 0;">
        <li>Use the Event Log to view detailed runtime information.</li>
        <li>Track execution status, row counts, errors, warnings, and performance metrics.</li>
      </ul>
    </div>
  </div>
</div>


## H. Scheduling the Pipeline

Once your pipeline is moved to production, the next step is to schedule it to run automatically.

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">

<style>
.tab-btn9 { padding:10px 24px; border:none; border-bottom:3px solid transparent; background:none; font-size:12pt; font-weight:700; color:#618794; cursor:pointer; transition:color 0.2s; font-family:'Segoe UI',sans-serif; }
.tab-btn9.active-tab9 { color:#2272B4; border-bottom:3px solid #2272B4; }
.tab-panel9 { display:none; }
.tab-panel9.active-panel9 { display:block; }
</style>

<div style="border-bottom:2px solid #EEEDE9; margin:20px 0 0 0; display:flex;">
  <button class="tab-btn9 active-tab9" onclick="showTab9(1)">Triggered Mode</button>
  <button class="tab-btn9" onclick="showTab9(2)">Continuous Mode</button>
</div>
<br>

<!-- Triggered -->
<div class="tab-panel9 active-panel9" id="tab9-1">
  <div style="border-radius:12px; overflow:hidden; border:2px solid #2272B4; box-shadow:0 3px 12px rgba(34,114,180,0.12);">
    <div style="background:#2272B4; padding:14px 22px; color:white; font-weight:800; font-size:12pt;">
      Triggered Mode
      <span style="background:white; color:#2272B4; font-size:8.5pt; font-weight:800; padding:3px 10px; border-radius:999px; margin-left:10px;">Ideal for batch workloads</span>
    </div>
    <div style="padding:22px 24px; background:#F9F7F4; display:grid; grid-template-columns:1fr 1fr; gap:14px;">
      <div style="background:white; border-radius:8px; border-left:4px solid #2272B4; padding:14px 16px;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">How It Works</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          Pipelines can be triggered <strong style="color:#0B2026;">manually</strong> or set to run on a
          <strong style="color:#0B2026;">recurring schedule</strong> (e.g., every 15 minutes, hourly, daily).
        </div>
      </div>
      <div style="background:white; border-radius:8px; border-left:4px solid #2272B4; padding:14px 16px;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">What Happens</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          The pipeline refreshes selected tables using the data
          <strong style="color:#0B2026;">available at the start of execution</strong>. Once updates are complete, the pipeline
          <strong style="color:#0B2026;">stops</strong>.
        </div>
      </div>
      <div style="background:white; border-radius:8px; border-left:4px solid #00A972; padding:14px 16px; grid-column:1/-1;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">Best For</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          Batch processing or pipelines that do not need to run constantly. The pipeline runs, completes its work, and stops —
          making it cost-efficient for scheduled, non-continuous workloads.
        </div>
      </div>
    </div>
  </div>
</div>

<!-- Continuous -->
<div class="tab-panel9" id="tab9-2">
  <div style="border-radius:12px; overflow:hidden; border:2px solid #00A972; box-shadow:0 3px 12px rgba(0,169,114,0.12);">
    <div style="background:#00A972; padding:14px 22px; color:white; font-weight:800; font-size:12pt;">
      Continuous Mode
      <span style="background:white; color:#00A972; font-size:8.5pt; font-weight:800; padding:3px 10px; border-radius:999px; margin-left:10px;">Ideal for streaming workloads</span>
    </div>
    <div style="padding:22px 24px; background:#F9F7F4; display:grid; grid-template-columns:1fr 1fr; gap:14px;">
      <div style="background:white; border-radius:8px; border-left:4px solid #00A972; padding:14px 16px;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">How It Works</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          Pipelines <strong style="color:#0B2026;">continuously process new data</strong> to keep streaming tables and materialized
          views up to date in near real-time.
        </div>
      </div>
      <div style="background:white; border-radius:8px; border-left:4px solid #00A972; padding:14px 16px;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">What Happens</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          The system <strong style="color:#0B2026;">monitors dependencies</strong> and updates only when source data changes —
          ensuring efficiency without unnecessary reprocessing.
        </div>
      </div>
      <div style="background:white; border-radius:8px; border-left:4px solid #FFAB00; padding:14px 16px; grid-column:1/-1;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px;">Best For</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">
          Streaming use cases where <strong style="color:#0B2026;">freshness and responsiveness are critical</strong>. The pipeline
          stays alive and processes data as it arrives — keeping results current at all times.
        </div>
      </div>
    </div>
  </div>
</div>

<script>
function showTab9(n) {
  document.querySelectorAll('.tab-btn9').forEach(function(t,i){
    t.classList.toggle('active-tab9', i===n-1);
  });
  document.querySelectorAll('.tab-panel9').forEach(function(p,i){
    p.classList.toggle('active-panel9', i===n-1);
  });
}
</script>

</div>

## I. Email Notifications
<br>

<div style="max-width:700px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
  <div style="display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;">
    <!-- Left: Text -->
    <div style="flex:1 1 0; min-width:260px; font-size:15px; color:#0B2026; line-height:1.7;">
      <br>
      As part of scheduling your pipeline, you can configure email notifications to keep stakeholders informed of pipeline activity.
      <br><br>
      Notifications can be set for any combination of three events.
    </div>
    <!-- Right: Image -->
    <div style="flex:1 1 0; min-width:260px; text-align:left;">
      <img
        src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_deploying_pipeline_production/email_notification_setting.png"
        alt="Email notification settings"
        style="max-width:100%; height:auto; border-radius:4px;">
    </div>
  </div>
</div>

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
<div style="margin:20px 0;">
  <div style="background:#F9F7F4; border:2px solid #EEEDE9; border-radius:12px; padding:24px; margin-bottom:16px;">
    <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:18px; padding-bottom:10px; border-bottom:2px solid #EEEDE9;">Configure Notifications for Any Combination of These Events</div>
    <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:12px;">
      <div style="background:white; border-radius:10px; border:2px solid #2272B4; overflow:hidden;">
        <div style="background:#2272B4; padding:10px 16px; font-weight:800; font-size:10pt; color:white; text-align:center;">Pipeline Starts</div>
        <div style="padding:14px 16px; font-size:10pt; color:#5A6F77; line-height:1.75; text-align:center;">
          Receive an alert the moment a pipeline run begins — useful for tracking execution timing.
        </div>
      </div>
      <div style="background:white; border-radius:10px; border:2px solid #00A972; overflow:hidden;">
        <div style="background:#00A972; padding:10px 16px; font-weight:800; font-size:10pt; color:white; text-align:center;">Pipeline Succeeds</div>
        <div style="padding:14px 16px; font-size:10pt; color:#5A6F77; line-height:1.75; text-align:center;">
          Get confirmation when a run completes successfully — gives confidence that data is fresh.
        </div>
      </div>
      <div style="background:white; border-radius:10px; border:2px solid #98102A; overflow:hidden;">
        <div style="background:#98102A; padding:10px 16px; font-weight:800; font-size:10pt; color:white; text-align:center;">Pipeline Fails</div>
        <div style="padding:14px 16px; font-size:10pt; color:#5A6F77; line-height:1.75; text-align:center;">
          Be immediately alerted to failures — enables fast response and minimizes data freshness gaps.
        </div>
      </div>
    </div>
    <div style="margin-top:14px; background:#FFAB00; border-radius:8px; padding:14px 18px; font-size:10.5pt; color:#98102A; line-height:1.75; text-align:center;">
      You can choose <strong style="color:white;">any combination</strong> of these three events — or all three — depending on your monitoring needs.
    </div>
  </div>
</div>
</div>

## J. Monitoring with the Pipeline Event Log
<br>

<div style="max-width:900px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
  <div style="display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;">
    <!-- Left: Text -->
    <div style="flex:1 1 0; min-width:260px; font-size:15px; color:#0B2026; line-height:1.7;">
      Another essential task when running pipelines in production is monitoring their health and performance.
    </div>
    <!-- Right: Image -->
    <div style="flex:1 1 0; min-width:600px; text-align:left;">
      <img
        src="https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/lecture_deploying_pipeline_production/monitoring_event_log.png"
        alt="Monioring event log"
        style="max-width:100%; height:auto; border-radius:4px;">
    </div>
  </div>
</div>

### J1. Pipeline Event Log

Spark Declarative Pipelines provide a pipeline event log that captures all critical information, including:

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
<div style="margin:20px 0;">
  <div style="background:#F9F7F4; border-radius:12px; border:2px solid #EEEDE9; overflow:hidden; margin-bottom:16px;">
    <div style="display:grid; grid-template-columns:repeat(2,1fr); gap:0;">
      <div style="padding:18px 20px; border-right:2px solid #EEEDE9; border-bottom:2px solid #EEEDE9;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px; padding-bottom:6px; border-bottom:2px solid #EEEDE9;">Audit Logs</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">Track <strong style="color:#1B3139;">who did what and when</strong> within the pipeline. Provides a full history of actions taken against your pipeline and its datasets.</div>
      </div>
      <div style="padding:18px 20px; border-bottom:2px solid #EEEDE9;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px; padding-bottom:6px; border-bottom:2px solid #EEEDE9;">Data Quality Checks</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">Monitor the <strong style="color:#1B3139;">results of any expectations or constraints</strong> applied during processing — including pass/fail counts and violation details.</div>
      </div>
      <div style="padding:18px 20px; border-right:2px solid #EEEDE9;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px; padding-bottom:6px; border-bottom:2px solid #EEEDE9;">Pipeline Progress</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">See the <strong style="color:#1B3139;">status and progress of each pipeline run</strong> — including row counts processed, stages completed, and current execution state.</div>
      </div>
      <div style="padding:18px 20px;">
        <div style="font-weight:800; font-size:10.5pt; color:#0B2026; margin-bottom:6px; padding-bottom:6px; border-bottom:2px solid #EEEDE9;">Data Lineage</div>
        <div style="font-size:10pt; color:#5A6F77; line-height:1.75;">Understand <strong style="color:#1B3139;">how data flows and transforms</strong> through your pipeline — from source ingestion through bronze, silver, and gold layers.</div>
      </div>
    </div>
  </div>
</div>
</div>

This comprehensive event log helps you quickly diagnose issues, ensure data integrity, and maintain full visibility into your pipeline operations.

### J2. Querying the Declarative Pipeline Event Log
<br>

<div style="max-width:1000px; margin:0 auto; font-family:'Segoe UI',sans-serif;">
<div style="display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-bottom:16px;">
<div style="background:#F9F7F4; border-radius:12px; padding:20px 22px; border:2px solid #2272B4;">
    <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:14px; border-bottom:2px solid #EEEDE9; padding-bottom:10px;">Publish to Metastore (Recommended)</div>
    <div style="font-size:10.5pt; color:#1B3139; line-height:1.8;">
      Publish the event log as a UC table using the advanced settings
      <ul>
      <li>Specify the table location (catalog, schema)
      <li>Define the table name
      </ul>
    </div>
  </div>
  <div style="background:#F9F7F4; border-radius:12px; padding:20px 22px; border:2px solid #EEEDE9;">
    <div style="font-weight:800; font-size:11pt; color:#0B2026; margin-bottom:14px; border-bottom:2px solid #EEEDE9; padding-bottom:10px;">Default Behavior</div>
    <div style="font-size:10.5pt; color:#1B3139; line-height:1.8;">
      By default, the event log is written as a <strong>hidden UC table</strong> located in the pipeline's default catalog and schema.
    </div>
  </div>
</div>
<div style="background:#F9F7F4; border:2px solid #EEEDE9; border-radius:12px; overflow:hidden; margin-bottom:16px;">
  <div style="background:#1B3139; padding:12px 20px; color:white; font-weight:800; font-size:10.5pt;">Querying the Event Log</div>
  <div style="padding:18px 22px;">
    <div style="font-size:10.5pt; color:#1B3139; line-height:1.75; margin-bottom:12px;">
      Once published, you can query the event log just like any other UC table, enabling you to build custom reports, dashboards, or perform deeper analysis on pipeline activity and health.
    </div>
<div class="code-block" data-language="sql">
SELECT * FROM &amp;lt;catalog&amp;gt;.&amp;lt;schema&amp;gt;.&amp;lt;event_log_table_name&amp;gt;
</div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
  document.querySelectorAll('.code-block').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var lang = block.getAttribute('data-language') || 'sql';
    var code = block.textContent.trim();
    var id = 'code-' + Math.random().toString(36).substr(2, 9);
    block.innerHTML = 
      '<div style="position:relative;margin:16px 0;">' +
        '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
        '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
      '</div>';
    var codeEl = document.getElementById(id);
    codeEl.textContent = code;
    Prism.highlightElement(codeEl);
    block.querySelector('.copy-btn').onclick = function() {
      var t = document.createElement('textarea');
      t.value = code;
      document.body.appendChild(t);
      t.select();
      document.execCommand('copy');
      document.body.removeChild(t);
      this.textContent = '✓ Copied!';
      setTimeout(() => this.textContent = 'Copy', 2000);
    };
  });
})();
</script>
  </div>
</div>
</div>
</div>

## K. Conclusion

In this lecture, you learned how streaming joins and production deployment work together in Apache Spark™ Declarative Pipelines:

- Streaming joins support different patterns: stream-snapshot joins for lookup enrichment, streaming table joins through materialized views, and stream-stream joins with windowing and watermarking.
- Choosing the right join type depends on whether sources are static or streaming, and whether the workflow needs enrichment or synchronization.
- Production deployment relies on scheduling, notifications, monitoring, and event logs for reliable, observable pipelines with audit logs, quality results, progress, and lineage.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/>
<a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> |
<a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> |
<a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
